# HogProf tutorial
This notebook includes all basic functionalities of the HogProf software: 

<ol>
  <li>extraction of phylogenetic profiles of protein (sub)families</li>
  <li>significance threshold calculation (optional, used for subfamilies)</li>
  <li>search for coevolving protein (sub)families</li>
  <li>network visualisation of the results</li>
</ol>

In [15]:
### imports
import os
from os.path import join, isfile
import pandas as pd
import sys
### importing HogProf
sys.path.append("../src")
from hogprof.utils import phylo
from hogprof.cli import main

In [2]:
### paths
# input
test_input = "../tests/data/micro_birds"
test_tree = join(test_input, "species_tree.nwk")
test_orthoxmls_dir = join(test_input, "splits")
# 1 profiles
test_output_main = join(test_input, "profiles")
test_output_levels = join(test_input, "levels_profiles")
# 2 thresholds
test_Ts_dir = join(test_output_levels, 'thresholds')

## Input files
HogProf requires two kinds of input to run. A <b>species tree</b> and <b>Hierarchical Orthologous Groups (HOGs)</b>. The latter can be obtained from one of two sources:
- precomputed <b>OMA Browser</b> database (as <code>hdf5</code>), available from download on the [OMA Browser](https://omabrowser.org/All/OmaServer.h5).
- [<b>FastOMA</b>](https://github.com/DessimozLab/FastOMA/tree/main) analysis result of a custom dataset (as <code>orthoxml</code> files). 

In this tutorial, only the FastOMA case will be be covered.

<b>NOTE</b>:The <code>FastOMA_HOGs.orthoxml</code> file that is being created as part of the FastOMA result needs to be split into multiple files (one per rootHOG) in order to run HogProf. This can be achieved with the [<code>orthoxml-tools</code>](https://github.com/DessimozLab/orthoxml-tools) toolkit using the command:

 <code>
 orthoxml-tools split --infile <var>FastOMA_HOGs.orthoxml</var> --outfile <var>splits</var> 
 </code>

For the test dataset used in the present notebook, the split <code>orthoxml</code> files are provided. Let's have a look at the files

In [3]:
### parse and print tree
tree = phylo.from_file(test_tree)
print(f"The dataset contains {len(tree.get_leaves())} species")
print(tree)

The dataset contains 10 species

      /-Crocodylus_porosus
     |
     |                     /-Aptenodytes_forsteri
     |                  /-|
     |               /-|   \-Oreotrochilus_melanogaster
     |              |  |
     |            /-|   \-Grus_americana
   /-|           |  |
  |  |         /-|   \-Taeniopygia_guttata
  |  |        |  |
  |  |      /-|   \-Columba_livia
  |  |     |  |
--|  |   /-|   \-Phoenicopterus_ruber
  |  |  |  |
  |   \-|   \-Gallus_gallus
  |     |
  |      \-Struthio_camelus
  |
   \-Homo_sapiens


In [ ]:
### parse and show info of orthoxmls
orthoxml_files = [p for p in os.listdir(test_orthoxmls_dir)
 if (isfile(join(test_orthoxmls_dir, p)) and 
     p.endswith(".orthoxml"))]
print(f"There are {len(orthoxml_files)} orthoxml files (rootHOGs) in the dataset:")
filecounter=0
for f in orthoxml_files:
    print(f)
    filecounter+=1
    if filecounter == 10:
        print("...")
        break

There are 4 orthoxml files (rootHOGs) in the dataset:
HOG:0015853_1.orthoxml
HOG:0016019_4.orthoxml
HOG:0014531_1.orthoxml
HOG:0016206_1.orthoxml


## 1: Phylogenetic profiles
### Protein families (HOGs)
HogProf uses the information in the input files to calculate phylogenetic profiles for each rootHOG. They are informed by on extant species and inferred ancestral events and are based on presence/absence of proteins, duplication events and loss events. 

<b>NOTE</b>: It is possible to change the parameters and consider e.g. only duplications or only losses, but all events are considered by default.

The <code>lshbuilder</code> script is responsible for this step.

In the command line, the syntax should be as follows:

<code>
lshbuilder 
--OrthoGlob 'input_dir/*.orthoxml' 
--mastertree input_dir/species_tree.nwk 
--outpath output_dir 
--specieslim 7 
</code>

<b>NOTE</b>: we lower the species limit <code>specieslim</code> from 10 to <b>7</b> here because we only have 10 species in the test dataset.

Let's run it in a code block here.

In [5]:
### the main function of HogProf is lshbuilder
main([
    "--OrthoGlob", f"{test_orthoxmls_dir}/*.orthoxml",
    "--mastertree", str(test_tree),
    "--outpath", str(test_output_main),
    "--specieslim", "7"
])

╭────────────────────────────────────────────────────────────────────────────────╮
│                                                                                │
│  ┌─┼─┐ HogProf v0.2.0                                                          │
│  ■ □ ■                                                                         │
│                                                                                │
│  Cite                   ]8;id=704909;https://doi.org/10.1371/journal.pcbi.1007553\Moi et al. (2020), PLOS Comp Biol]8;;\                      │
│                                                                                │
│  Input                  OrthoXML: ../tests/data/micro_birds/splits/*.orthoxml  │
│  Output                 ../tests/data/micro_birds/profiles                     │
│  Workers                1                                                      │
│                                                                                │
╰────────────────────────────────────────────────────────────────────────────────╯

Initializing...

Using orthoxml files from: ../tests/data/micro_birds/splits/*.orthoxml

Initializing LSHBuilder


/home/agavriil/miniforge3/envs/hogprof_v020/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[17:12:46] INFO     making tree weights with 19 taxa

...

           INFO     Processing OrthoXML files: done (4 items in 0.0s)

           INFO     Input queued; waiting for workers to finish

           INFO     Workers finished; finalizing saver

[17:12:47] INFO     Pipeline complete

           INFO     Done in 0.81s

Let's take a look at the result files.

In [6]:
### parse and print result files
lshbuilder_files = [p for p in os.listdir(test_output_main)]
print(f"There are {len(lshbuilder_files)} files in the lshbuilder output folder:")
for f in lshbuilder_files:
    print(f)

There are 6 files in the lshbuilder output folder:
taxaIndex.pkl
newlshforest.pkl
wmg.pkl
master_tree.corrected.nwk
fam2orthoxml.csv
hashes.h5


In [7]:
### take a look at the fam2orthoxml table
main_fam2orthoxml = join(test_output_main, 'fam2orthoxml.csv')
profiles_main_df = pd.read_csv(main_fam2orthoxml, header=0, names=['fam','ortho'])
print(f"This table contains the fam (HOG) indices and ortho (file) of the \n"
      f"{profiles_main_df.shape[0]} HOGs whose profiles were extracted")
profiles_main_df.head()

This table contains the fam (HOG) indices and ortho (file) of the 
3 HOGs whose profiles were extracted


,fam,ortho
0,0,../tests/data/micro_birds/splits/HOG:0015853_1...
1,1,../tests/data/micro_birds/splits/HOG:0016019_4...
2,3,../tests/data/micro_birds/splits/HOG:0016206_1...


### Protein subfamilies (subHOGs)
You can use the <code>slicesubhogs</code> argument to extract phylogenetic profiles for each subHOG, i.e. subfamily instead. This means that instead of extracting one profile per rootHOG (orthoxml file), multiple profiles will calculated, at each taxonomic level, as long as the minimum requirements are fulfilled (by default, a profile is calculated if the HOG/subHOG contains 10 species with a protein and 10 proteins in total). 

In the command line, the syntax should be as follows:

<code>
lshbuilder 
--OrthoGlob 'input_dir/*.orthoxml' 
--mastertree input_dir/species_tree.nwk 
--outpath output_dir 
--specieslim 7 
--slicesubhogs
</code>

Let's run it in a code block here.

In [8]:
### the main function of HogProf is lshbuilder
main([
    "--OrthoGlob", f"{test_orthoxmls_dir}/*.orthoxml",
    "--mastertree", str(test_tree),
    "--outpath", str(test_output_levels),
    "--specieslim", "7",
    "--slicesubhogs"
])

╭────────────────────────────────────────────────────────────────────────────────╮
│                                                                                │
│  ┌─┼─┐ HogProf v0.2.0                                                          │
│  ■ □ ■                                                                         │
│                                                                                │
│  Cite                   ]8;id=704911;https://doi.org/10.1371/journal.pcbi.1007553\Moi et al. (2020), PLOS Comp Biol]8;;\                      │
│                                                                                │
│  Input                  OrthoXML: ../tests/data/micro_birds/splits/*.orthoxml  │
│  Output                 ../tests/data/micro_birds/levels_profiles              │
│  Workers                1                                                      │
│                                                                                │
╰────────────────────────────────────────────────────────────────────────────────╯

Initializing...

Using orthoxml files from: ../tests/data/micro_birds/splits/*.orthoxml

Initializing LSHBuilder


[17:12:47] INFO     making tree weights with 19 taxa

Kept dataset nodes: None

Setting up input data reader


...

           INFO     Processing OrthoXML files: done (4 items in 0.0s)

           INFO     Input queued; waiting for workers to finish

           INFO     Workers finished; finalizing saver

           INFO     Pipeline complete

           INFO     Done in 0.66s

In [9]:
### parse and print result files
lshbuilder_levels_files = [p for p in os.listdir(test_output_levels)]
print(f"There are {len(lshbuilder_levels_files)} files in the lshbuilder levels output folder:")
for f in lshbuilder_levels_files:
    print(f)

There are 6 files in the lshbuilder levels output folder:
taxaIndex.pkl
newlshforest.pkl
wmg.pkl
master_tree.corrected.nwk
fam2orthoxml.csv
hashes.h5


In [10]:
### take a look at the fam2orthoxml table
levels_fam2orthoxml = join(test_output_levels, 'fam2orthoxml.csv')
profiles_levels_df = pd.read_csv(levels_fam2orthoxml)
print(f"This table contains the fam (subHOG) indices and ortho (file) of the \n"
      f"{profiles_levels_df.shape[0]} subHOGs whose profiles were extracted")
profiles_levels_df.head()

This table contains the fam (subHOG) indices and ortho (file) of the 
26 subHOGs whose profiles were extracted


,fam,subhog_id,ortho
0,0,_HOG:0015853_1,../tests/data/micro_birds/splits/HOG:0015853_1...
1,0,Archosauria_HOG:0015853.1g_2,../tests/data/micro_birds/splits/HOG:0015853_1...
2,0,Aves_HOG:0015853.1g_4,../tests/data/micro_birds/splits/HOG:0015853_1...
3,0,Archosauria_HOG:0015853.1h_2,../tests/data/micro_birds/splits/HOG:0015853_1...
4,0,Aves_HOG:0015853.1h_4,../tests/data/micro_birds/splits/HOG:0015853_1...


## 2: Significance thresholds
<b>NOTE</b>: this functionality is relevant for the <code>slicesubhogs</code> mode (subfamilies).

Since version <code>0.2.0</code>, HogProf includes a script for generating distributions of Jaccard values for each taxonomic level, in order to calculate an empirical threshold of significance (a=0.05) for that level.

The <code>generate_jaccard_dist</code> script is responsible for this step.

In the command line, the syntax should be as follows:

<code>
python ../src/hogprof/generate_jaccard_distr.py 
--hogprof_folder output_dir/hogprof_output 
--mastertree input_dir/species_tree.nwk 
--output thresholds_dir 
--expected_root Amniota
</code>

Let's run it in a code block here.

In [14]:
### TBD: update to newest 
#generate_jaccard_distr.main(hogprofoutputfolder=test_output_levels,
#                            outputdir=test_Ts_dir,
#                            profiler_path=None,
#                            expected_root='Amniota')

Let's look at the files created. The most important file is <code>bins_jaccard_thresholds.csv</code>, which contains the empirical thresholds for each taxonomic level considered. These can be used to extract significantly coevolving pairs of subHOGs at given levels.

In [ ]:
thresholds_files = [p for p in os.listdir(test_orthoxmls_dir) 
                    if isfile(join(test_orthoxmls_dir, p))]
thresholds_table = join(test_Ts_dir, 'bins_jaccard_thresholds.csv')
npy_files = [p for p in thresholds_files
             if p.endswith(".npy")]
print(f"Taxonomic levels considered: {len(npy_files)}")
ts_df = pd.read_csv(thresholds_table)
ts_df.head()

A figure representing the thresholds in different taxonomic levels is also generated:

In [ ]:
from IPython.display import Image, display
thresholds_png = join(test_Ts_dir, 'bin_vs_jaccard_thresholds.png')
display(Image(filename=thresholds_png))

## 3: Similar profiles
### Protein families (HOGs)
HogProf is designed to be scalable. The profiles calculated in step 1 are stored in such a way that identifying highly similar profiles is very fast.